# PathJev-P: ask a histopathology patch a question, get probabilities

**Research demo. Not a medical device and not for diagnosis.** PathJev-P is over-confident on organs it was not
trained on, so a high probability does not mean a reliable answer.

PathJev-P is MedGemma-1.5-4B with a LoRA adapter and a decision head. For one image it returns, in a single forward
pass, a probability distribution over the candidate answers you write, plus an optional "other tissue / none of
these" candidate.

**Before you start**
1. Choose *Runtime → Change runtime type → T4 GPU* (free).
2. On Hugging Face, open https://huggingface.co/google/medgemma-1.5-4b-it and accept its terms (Health AI Developer
   Foundations).
3. Create a *read* access token at https://huggingface.co/settings/tokens.

Model: https://huggingface.co/Cristiandieti/PathJev-P

In [ ]:
!pip install -q -U "transformers>=4.56" "peft>=0.17" bitsandbytes accelerate

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste your own read token; it is not stored in the notebook

In [ ]:
import sys, torch
from huggingface_hub import snapshot_download
REPO_ID = "Cristiandieti/PathJev-P"
local = snapshot_download(REPO_ID)          # adapter, head, inference code, three example patches
sys.path.insert(0, local)
from pathjev_infer import PathJev
assert torch.cuda.is_available(), "select a GPU runtime (T4)"
model = PathJev.load(local, token=True)       # 4-bit NF4 on the GPU, as in the paper

## Ask a question
One candidate per line. Leave `ADD_OTHER = True` to include "other tissue / none of these".

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def ask(image_path, question, candidates, add_other=True):
    cands = [{"id": f"c{i}", "text": t.strip()} for i, t in enumerate(candidates.strip().splitlines()) if t.strip()]
    p = model.predict(image_path, [{"instructions": question, "candidates": cands}], add_other=add_other)[0]
    names = {c["id"]: c["text"] for c in cands} | {"other": "other tissue / none of these"}
    labels, values = [names[k] for k in p], list(p.values())
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.6]})
    a.imshow(Image.open(image_path).convert("RGB")); a.axis("off")
    b.barh(labels[::-1], values[::-1], color="#eb6834"); b.set_xlim(0, 1); b.set_xlabel("probability")
    b.set_title(question, fontsize=10); plt.tight_layout(); plt.show()
    return dict(zip(labels, values))

# example patch (SICAPv2, CC BY 4.0; expert annotation: Gleason pattern 4)
ask(f"{local}/examples/sicapv2_gp4.jpg",
    "Which Gleason pattern predominates in this prostate tissue?",
    """benign (non-cancerous) prostate tissue
Gleason pattern 3 carcinoma
Gleason pattern 4 carcinoma
Gleason pattern 5 carcinoma""")

## Your own patch
Upload an H&E patch (ideally 10–20×, a few hundred pixels). It is resized to 896×896.

In [ ]:
from google.colab import files
up = files.upload()
path = next(iter(up))
ask(path, "Is carcinoma present in this tissue?", "no\nyes", add_other=False)

## Notes
- Several questions can be asked in one call: pass a list of questions to `model.predict`.
- Results of the paper: higher balanced accuracy than a generative model trained on the same data, mostly on organs
  unseen in training. The calibration advantage depends on how the generative model is read out. See the model card.
- Use is subject to the MedGemma terms (Health AI Developer Foundations) and to the licence of this model repo.